# 02 合并清洗后高频数据（ETF + 债券）

目标：
- 仅使用清洗后的单文件数据（`cleaned_single_files/*.parquet`）进行合并；
- 按标的（511090、511130、019742、019776、019789）分别纵向拼接月度文件；
- 做严格的 schema/timestamp/重复文件诊断；
- 输出每个标的的 merged CSV 与 diagnostics 报告。


In [ ]:
from __future__ import annotations
from pathlib import Path
import re
import hashlib
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)


In [ ]:
# =========================
# 1) 配置
# =========================
CLEANED_DIR = Path('cleaned_single_files')
MERGED_DIR = Path('merged_outputs')
REPORT_DIR = Path('reports_merge')
MERGED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

INSTRUMENTS = ['511090', '511130', '019742', '019776', '019789']
TIME_COL = 'trade_time'

# 标的 -> 期望输出时间区间（来自已有预分析，可用于校验）
EXPECTED_RANGES = {
    '511090': ('20250701', '20251231'),
    '511130': ('20260101', '20260228'),
    '019742': ('20260101', '20260312'),
    '019776': ('20260101', '20260312'),
    '019789': ('20260101', '20260312'),
}


In [ ]:
# =========================
# 2) 工具函数
# =========================
def file_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def infer_instrument_from_name(name: str) -> str | None:
    m = re.match(r'^(\d{6})', name)
    return m.group(1) if m else None

def load_cleaned_files(cleaned_dir: Path) -> pd.DataFrame:
    files = sorted(cleaned_dir.glob('*_cleaned.parquet'))
    rec = []
    for p in files:
        inst = infer_instrument_from_name(p.name)
        rec.append({
            'path': str(p),
            'file_name': p.name,
            'instrument': inst,
            'file_size_bytes': p.stat().st_size,
            'sha256': file_sha256(p),
        })
    return pd.DataFrame(rec)

def normalize_one_df(df: pd.DataFrame, file_name: str, time_col: str = 'trade_time'):
    diag = {'file_name': file_name, 'rows_before': len(df), 'has_trade_time': time_col in df.columns}
    out = df.copy()
    if time_col not in out.columns:
        diag.update({'nat_count': None, 'dup_ts_before': None, 'rows_after': len(out), 'is_monotonic': None})
        return out, diag
    out[time_col] = pd.to_datetime(out[time_col], errors='coerce')
    nat_count = int(out[time_col].isna().sum())
    out = out.dropna(subset=[time_col]).copy()
    out = out.sort_values(time_col)
    dup_subset = ['code', time_col] if 'code' in out.columns else [time_col]
    dup_ts_before = int(out.duplicated(subset=dup_subset).sum())
    out = out.drop_duplicates(subset=dup_subset, keep='last').copy()
    diag.update({
        'nat_count': nat_count,
        'dup_ts_before': dup_ts_before,
        'rows_after': len(out),
        'is_monotonic': bool(out[time_col].is_monotonic_increasing),
        'ts_min': out[time_col].min() if len(out) else pd.NaT,
        'ts_max': out[time_col].max() if len(out) else pd.NaT,
    })
    return out, diag

def schema_signature(df: pd.DataFrame) -> tuple:
    return tuple(df.columns.tolist())

def align_schema(dfs: dict[str, pd.DataFrame]):
    all_cols = sorted(set().union(*[set(df.columns) for df in dfs.values()])) if dfs else []
    aligned = {}
    schema_rec = []
    for fname, df in dfs.items():
        missing = [c for c in all_cols if c not in df.columns]
        extra = [c for c in df.columns if c not in all_cols]
        tmp = df.copy()
        for c in missing:
            tmp[c] = np.nan
        tmp = tmp[all_cols]
        aligned[fname] = tmp
        schema_rec.append({'file_name': fname, 'col_count_before': df.shape[1], 'col_count_after': tmp.shape[1], 'missing_columns': '|'.join(missing), 'extra_columns': '|'.join(extra)})
    return aligned, pd.DataFrame(schema_rec), all_cols

def detect_duplicate_pairs(file_info_df: pd.DataFrame, instrument: str) -> pd.DataFrame:
    sub = file_info_df[file_info_df['instrument'] == instrument].copy()
    if sub.empty:
        return sub
    sub['dup_group'] = sub.groupby(['file_size_bytes', 'sha256'], dropna=False).ngroup()
    out = []
    for gid, g in sub.groupby('dup_group'):
        action_list = ['keep'] + ['drop'] * (len(g)-1) if len(g) > 1 else ['keep']
        for (_, row), action in zip(g.iterrows(), action_list):
            out.append({
                'instrument': instrument,
                'dup_group': gid,
                'file_name': row['file_name'],
                'file_size_bytes': row['file_size_bytes'],
                'sha256': row['sha256'],
                'action': action,
                'dup_count_in_group': len(g),
            })
    return pd.DataFrame(out).sort_values(['dup_group', 'file_name'])

def unify_dtypes(df: pd.DataFrame, time_col='trade_time'):
    out = df.copy()
    for c in out.columns:
        if c in [time_col, 'code', 'trading_phase_code']:
            continue
        out[c] = pd.to_numeric(out[c], errors='ignore')
    return out

def merge_one_instrument(inst: str, file_info_df: pd.DataFrame):
    dup_report = detect_duplicate_pairs(file_info_df, inst)
    keep_files = dup_report.loc[dup_report['action'] == 'keep', 'file_name'].tolist()

    dfs_raw = {}
    ts_diag = []
    for fn in keep_files:
        p = CLEANED_DIR / fn
        if not p.exists():
            continue
        df = pd.read_parquet(p)
        df, d = normalize_one_df(df, file_name=fn, time_col=TIME_COL)
        df = unify_dtypes(df, time_col=TIME_COL)
        dfs_raw[fn] = df
        ts_diag.append(d)

    if not dfs_raw:
        return None, pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    aligned, schema_diag, final_cols = align_schema(dfs_raw)

    cat = pd.concat([aligned[k] for k in sorted(aligned.keys())], axis=0, ignore_index=True)
    pre_drop_dup = len(cat)
    cat = cat.sort_values(TIME_COL)
    dup_after_concat = int(cat.duplicated(subset=[TIME_COL]).sum()) if TIME_COL in cat.columns else None
    if TIME_COL in cat.columns:
        cat = cat.drop_duplicates(subset=[TIME_COL], keep='last').copy()

    merge_diag = pd.DataFrame([{
        'instrument': inst,
        'files_kept': len(keep_files),
        'rows_before_concat': int(sum(len(v) for v in aligned.values())),
        'rows_after_concat_before_dropdup': pre_drop_dup,
        'dup_ts_after_concat': dup_after_concat,
        'rows_final': len(cat),
        'col_final': cat.shape[1],
        'ts_min': cat[TIME_COL].min() if TIME_COL in cat.columns else pd.NaT,
        'ts_max': cat[TIME_COL].max() if TIME_COL in cat.columns else pd.NaT,
        'is_monotonic': bool(cat[TIME_COL].is_monotonic_increasing) if TIME_COL in cat.columns else None,
        'memory_mb': cat.memory_usage(deep=True).sum() / (1024**2),
    }])

    missing_summary = pd.DataFrame({
        'column': cat.columns,
        'missing_count': [int(cat[c].isna().sum()) for c in cat.columns],
        'missing_ratio': [float(cat[c].isna().mean()) for c in cat.columns],
        'instrument': inst,
    })

    ts_diag_df = pd.DataFrame(ts_diag)
    return cat, dup_report, schema_diag, ts_diag_df, merge_diag, missing_summary


In [ ]:
# =========================
# 3) 扫描 cleaned 数据并分标的合并
# =========================
file_info_df = load_cleaned_files(CLEANED_DIR)
display(file_info_df.head())

all_dup_reports = []
all_schema_reports = []
all_ts_reports = []
all_merge_reports = []
all_missing_reports = []

merged_outputs = {}

for inst in INSTRUMENTS:
    print(f"\n===== Merge instrument: {inst} =====")
    result = merge_one_instrument(inst, file_info_df)
    if result[0] is None:
        print(f"[WARN] No cleaned files found for {inst}")
        continue

    merged_df, dup_report, schema_diag, ts_diag_df, merge_diag, missing_summary = result
    merged_outputs[inst] = merged_df

    all_dup_reports.append(dup_report)
    all_schema_reports.append(schema_diag.assign(instrument=inst))
    all_ts_reports.append(ts_diag_df.assign(instrument=inst))
    all_merge_reports.append(merge_diag)
    all_missing_reports.append(missing_summary)

    # 终端验证输出
    print('shape:', merged_df.shape)
    if TIME_COL in merged_df.columns:
        print('trade_time range:', merged_df[TIME_COL].min(), '->', merged_df[TIME_COL].max())
        print('is_monotonic_increasing:', merged_df[TIME_COL].is_monotonic_increasing)
        print('dup trade_time count:', merged_df.duplicated(subset=[TIME_COL]).sum())
    display(merged_df.head(3))


In [ ]:
# =========================
# 4) 保存 merged CSV + diagnostics
# =========================
dup_report_all = pd.concat(all_dup_reports, ignore_index=True) if all_dup_reports else pd.DataFrame()
schema_report_all = pd.concat(all_schema_reports, ignore_index=True) if all_schema_reports else pd.DataFrame()
ts_report_all = pd.concat(all_ts_reports, ignore_index=True) if all_ts_reports else pd.DataFrame()
merge_report_all = pd.concat(all_merge_reports, ignore_index=True) if all_merge_reports else pd.DataFrame()
missing_report_all = pd.concat(all_missing_reports, ignore_index=True) if all_missing_reports else pd.DataFrame()

for inst, df in merged_outputs.items():
    if TIME_COL in df.columns and len(df) > 0:
        start_date = pd.to_datetime(df[TIME_COL].min()).strftime('%Y%m%d')
        end_date = pd.to_datetime(df[TIME_COL].max()).strftime('%Y%m%d')
    else:
        start_date, end_date = EXPECTED_RANGES.get(inst, ('unknown', 'unknown'))
    out_name = f"{inst}_{start_date}_{end_date}_merged.csv"
    out_path = MERGED_DIR / out_name
    df.to_csv(out_path, index=False)
    print('saved:', out_path)

# diagnostics 导出
dup_report_all.to_csv(REPORT_DIR / 'duplicate_file_report.csv', index=False)
schema_report_all.to_csv(REPORT_DIR / 'schema_consistency_report.csv', index=False)
ts_report_all.to_csv(REPORT_DIR / 'timestamp_integrity_report.csv', index=False)
merge_report_all.to_csv(REPORT_DIR / 'merge_validation_report.csv', index=False)
missing_report_all.to_csv(REPORT_DIR / 'post_merge_missing_summary.csv', index=False)

print('diagnostics saved to', REPORT_DIR)


In [ ]:
# =========================
# 5) post-merge validation 快速总览
# =========================
display(merge_report_all)

for inst, df in merged_outputs.items():
    print(f"\n==== {inst} INFO ====")
    display(df.dtypes.rename('dtype').to_frame().head(20))
    print(df.info())
    with pd.option_context('display.max_columns', 50):
        display(df.describe(include='all').T.head(20))
